# Customer Churn — EDA, Cleaning & Model Training

**Adept Tech Solutions — AI Engineer Assessment (Stage 1)**

This notebook covers: exploratory data analysis, documented data-cleaning
decisions, preprocessing pipeline construction, training and evaluating a
baseline (Logistic Regression) against a comparison model (Random Forest),
a justified metric choice, model selection, and export of the artifacts
used by `predict_churn_risk()` outside this notebook (`src/model/predict.py`).

This notebook is written to run **standalone in Google Colab** — it clones
nothing and imports nothing project-specific except what it defines inline;
the same cleaning/training logic also lives in `src/model/` as importable
modules for the Streamlit app and agent built in later stages.


## 0. Setup

In [1]:
# If running in Colab, install any missing dependencies.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import sys
if IN_COLAB:
    !pip install -q scikit-learn pandas numpy joblib


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix
)
import joblib
import json
import os

pd.set_option("display.max_columns", None)
RANDOM_STATE = 42


In [3]:
# Load the dataset.
#
# In Colab: upload Customer-Churn.csv when prompted (Files pane, or the
# upload widget below). Locally / in this repo: it's already at
# data/raw_churn.csv relative to the project root.

CANDIDATE_PATHS = ["data/raw_churn.csv", "Customer-Churn.csv", "/content/Customer-Churn.csv"]
data_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)

if data_path is None and IN_COLAB:
    from google.colab import files
    print("Dataset not found locally — please upload Customer-Churn.csv")
    uploaded = files.upload()
    data_path = list(uploaded.keys())[0]

assert data_path is not None, "Could not locate Customer-Churn.csv"
print("Loading dataset from:", data_path)

raw_df = pd.read_csv(data_path)
print("Shape:", raw_df.shape)
raw_df.head()


Loading dataset from: data/raw_churn.csv
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 1. Exploratory Data Analysis

Starting from a clean look at structure, types, and target balance before
touching anything — no assumptions carried in from outside this notebook.


In [4]:
raw_df.dtypes


customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [5]:
print("customerID unique values:", raw_df["customerID"].nunique(), "/ rows:", len(raw_df))
print()
print("Duplicate rows (all columns, including customerID):", raw_df.duplicated().sum())
print()
print("Nulls per column:")
print(raw_df.isnull().sum())


customerID unique values: 7043 / rows: 7043

Duplicate rows (all columns, including customerID): 0

Nulls per column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [6]:
print("Churn class balance:")
print(raw_df["Churn"].value_counts(normalize=True))


Churn class balance:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


**Finding — class imbalance.** ~26.5% of customers churned. This is moderate
imbalance, not extreme, but it's enough that plain accuracy would be
misleading (a model that always predicts "No churn" scores ~73.5% accuracy
while being useless). This directly informs the metric choice in Section 4.


In [7]:
raw_df["tenure"].describe()


count    7043.000000
mean       32.371149
std        24.559481
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64

In [8]:
raw_df["MonthlyCharges"].describe()


count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: MonthlyCharges, dtype: float64

### 1.1 `TotalCharges` — investigating the type issue

In [9]:
# TotalCharges is loaded as a string/object column. Coerce to numeric and
# see what fails.
total_charges_numeric = pd.to_numeric(raw_df["TotalCharges"], errors="coerce")
blank_mask = total_charges_numeric.isnull()
print("Rows where TotalCharges is not a valid number:", blank_mask.sum())
raw_df.loc[blank_mask, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]


Rows where TotalCharges is not a valid number: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


In [10]:
# What is the actual raw string value in these cells?
print("Raw values (repr, to catch whitespace-only strings):")
for v in raw_df.loc[blank_mask, "TotalCharges"]:
    print(repr(v))

print()
print("All 11 blank rows have tenure == 0:", (raw_df.loc[blank_mask, "tenure"] == 0).all())
print("All tenure==0 rows are exactly these 11:",
      (raw_df["tenure"] == 0).sum() == blank_mask.sum())
print("Churn value among these 11 rows:", raw_df.loc[blank_mask, "Churn"].unique())


Raw values (repr, to catch whitespace-only strings):
' '
' '
' '
' '
' '
' '
' '
' '
' '
' '
' '

All 11 blank rows have tenure == 0: True
All tenure==0 rows are exactly these 11: True
Churn value among these 11 rows: <StringArray>
['No']
Length: 1, dtype: str


In [11]:
# Does TotalCharges actually represent accumulated billing (tenure * monthly)?
valid = raw_df[~blank_mask].copy()
valid["TotalCharges_num"] = total_charges_numeric[~blank_mask]
valid["implied_total"] = valid["MonthlyCharges"] * valid["tenure"]
print("Correlation(actual TotalCharges, MonthlyCharges * tenure):",
      valid["TotalCharges_num"].corr(valid["implied_total"]))


Correlation(actual TotalCharges, MonthlyCharges * tenure): 0.9995598572867933


**Finding & decision — `TotalCharges` blanks.**
The 11 blank rows are not random missingness or a data-entry error:
- All 11 have `tenure == 0` (and these are the *only* tenure=0 rows in the
  dataset), i.e. brand-new customers who joined but haven't been billed yet.
- None of them have churned — consistent with "just signed up."
- Across the rest of the dataset, `TotalCharges` correlates at **r ≈ 0.9996**
  with `MonthlyCharges × tenure`, confirming it represents accumulated
  billing over the customer's tenure.

Given that, the structurally correct value for a tenure=0 customer is
**0.0** — they haven't accumulated any charges yet. Filling with the column
mean or median would fabricate billing history that doesn't exist and would
quietly bias the numeric feature distribution. Dropping the 11 rows was also
considered and rejected: they're a real (if small) segment — brand-new
signups — and the agent built in later stages needs to be able to answer
questions about the full customer base, including day-one customers.


### 1.2 Other data-quality checks

In [12]:
# Leading/trailing whitespace in categorical values
for c in raw_df.columns:
    vals = raw_df[c].dropna().unique()
    bad = [v for v in vals if isinstance(v, str) and v != v.strip()]
    if bad:
        print(c, "has whitespace issues:", bad)
print("(no output above other than TotalCharges' blank-space entries = no other whitespace issues)")


TotalCharges has whitespace issues: [' ']
(no output above other than TotalCharges' blank-space entries = no other whitespace issues)


In [13]:
# customerID format sanity check
bad_ids = raw_df[~raw_df["customerID"].str.match(r"^\d{4}-[A-Z]{5}$")]
print("customerIDs not matching NNNN-AAAAA format:", len(bad_ids))


customerIDs not matching NNNN-AAAAA format: 0


In [14]:
# "No phone/internet service" categories — are they consistent with the
# parent service flag, or a sign of inconsistent labeling?
print("MultipleLines values when PhoneService == 'No':")
print(raw_df.loc[raw_df.PhoneService == "No", "MultipleLines"].value_counts())
print()
for col in ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]:
    vc = raw_df.loc[raw_df.InternetService == "No", col].value_counts()
    print(col, "when InternetService == 'No':", dict(vc))


MultipleLines values when PhoneService == 'No':
MultipleLines
No phone service    682
Name: count, dtype: int64

OnlineSecurity when InternetService == 'No': {'No internet service': np.int64(1526)}
OnlineBackup when InternetService == 'No': {'No internet service': np.int64(1526)}
DeviceProtection when InternetService == 'No': {'No internet service': np.int64(1526)}
TechSupport when InternetService == 'No': {'No internet service': np.int64(1526)}
StreamingTV when InternetService == 'No': {'No internet service': np.int64(1526)}
StreamingMovies when InternetService == 'No': {'No internet service': np.int64(1526)}


**Finding.** These "No phone service" / "No internet service" labels are
perfectly consistent with the parent flag — every row with `PhoneService ==
"No"` has `MultipleLines == "No phone service"`, and the same holds for all
six internet-dependent service columns. This is redundant information, not
a data error, so it doesn't need cleaning — it's a modeling consideration
(one-hot encoding will naturally absorb the redundancy).


In [15]:
# Duplicate check on FEATURES only (excluding customerID and the target),
# to check for duplicate customer *records* as opposed to duplicate IDs.
df_check = raw_df.copy()
df_check["TotalCharges"] = total_charges_numeric.fillna(0.0)
feature_cols = [c for c in df_check.columns if c not in ("customerID", "Churn")]
dup_mask = df_check.duplicated(subset=feature_cols, keep=False)
print("Rows sharing an identical feature profile with another row:", dup_mask.sum())
print("All of these have tenure == 1:", (df_check.loc[dup_mask, "tenure"] == 1).all())

dup_groups = df_check[dup_mask].copy()
dup_groups["group_key"] = dup_groups[feature_cols].astype(str).agg("|".join, axis=1)
mixed = dup_groups.groupby("group_key")["Churn"].nunique()
print("Groups of identical-feature rows:", mixed.shape[0])
print("Of those, groups with BOTH Yes and No churn outcomes:", (mixed > 1).sum())


Rows sharing an identical feature profile with another row: 73
All of these have tenure == 1: True
Groups of identical-feature rows: 33
Of those, groups with BOTH Yes and No churn outcomes: 18


**Finding — feature collisions with mixed labels (irreducible error).**
73 rows across 33 groups share an identical feature profile with at least
one other row once `customerID` and `Churn` are excluded. All 73 have
`tenure == 1`: at tenure=1, `TotalCharges` equals `MonthlyCharges` exactly
(removing a degree of freedom), and `MonthlyCharges` is effectively
quantized by plan-pricing combinations, so first-month customers on similar
plans can coincidentally land on an identical feature row. These are real,
distinct customers (different `customerID`s), not duplicate records — kept
as-is.

Notably, **18 of the 33 groups have mixed outcomes**: the identical feature
profile produced both "Yes" and "No" churn for different real customers.
This means no classifier — regardless of algorithm — can achieve a perfect
score on this dataset with these features; it sets a real ceiling on
achievable performance and partly explains why the models below land in the
0.84 ROC-AUC range rather than near 1.0. This is a property of the data,
not something to "fix."


### 1.3 Churn rate by segment

In [16]:
df_eda = raw_df.copy()
df_eda["TotalCharges"] = total_charges_numeric.fillna(0.0)
churn_bin = (df_eda["Churn"] == "Yes").astype(int)

print("Churn rate by Contract:")
print(df_eda.groupby("Contract", observed=True).apply(lambda g: (g.Churn == "Yes").mean(), include_groups=False))


Churn rate by Contract:
Contract
Month-to-month    0.427097
One year          0.112695
Two year          0.028319
dtype: float64


In [17]:
print("Churn rate by InternetService:")
print(df_eda.groupby("InternetService", observed=True).apply(lambda g: (g.Churn == "Yes").mean(), include_groups=False))


Churn rate by InternetService:
InternetService
DSL            0.189591
Fiber optic    0.418928
No             0.074050
dtype: float64


In [18]:
print("Churn rate by PaymentMethod:")
print(df_eda.groupby("PaymentMethod", observed=True).apply(lambda g: (g.Churn == "Yes").mean(), include_groups=False))


Churn rate by PaymentMethod:
PaymentMethod
Bank transfer (automatic)    0.167098
Credit card (automatic)      0.152431
Electronic check             0.452854
Mailed check                 0.191067
dtype: float64


In [19]:
print("Correlation of numeric features with churn:")
for c in ["tenure", "MonthlyCharges", "TotalCharges"]:
    print(f"  {c}: {df_eda[c].corr(churn_bin):.4f}")


Correlation of numeric features with churn:
  tenure: -0.3522
  MonthlyCharges: 0.1934
  TotalCharges: -0.1983


**Findings.** Churn is strongly associated with contract flexibility and
service type: month-to-month contracts churn at ~42.7% vs. ~2.8% for
two-year contracts; fiber-optic internet customers churn at ~41.9% vs.
~19.0% for DSL; electronic-check payers churn at ~45.3% vs. ~15–19% for
other payment methods. Tenure is negatively correlated with churn (-0.35) —
newer customers are more likely to leave. These patterns line up with the
model's learned coefficients in Section 5, which is a useful sanity check
that the model is picking up genuine signal rather than noise.


## 2. Cleaning

In [20]:
def load_and_clean_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Apply the documented cleaning decisions from Section 1."""
    df = raw_df.copy().set_index("customerID")
    tc_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
    blank = tc_numeric.isnull()
    if blank.any():
        assert (df.loc[blank, "tenure"] == 0).all(), \
            "blank TotalCharges rows found with tenure != 0 — re-investigate"
        df["TotalCharges"] = tc_numeric.fillna(0.0)
    else:
        df["TotalCharges"] = tc_numeric
    return df

df = load_and_clean_data(raw_df)
assert df["TotalCharges"].isnull().sum() == 0
print("Cleaned shape:", df.shape)
df.head()


Cleaned shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
customerID,,,,,,,,,,,,,,,,,,,,
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Preprocessing Pipeline

Numeric features (`tenure`, `MonthlyCharges`, `TotalCharges`, `SeniorCitizen`
— already 0/1) are standardized. All remaining columns are categorical and
one-hot encoded, with `drop="if_binary"` so genuinely binary columns (e.g.
`Partner`, `PhoneService`) don't get a redundant second dummy column, while
multi-category columns (e.g. `Contract`, `PaymentMethod`) keep a full set of
indicator columns. The split happens **before** any fitting, and the
preprocessor is fit only on the training split (inside the `Pipeline`), to
avoid leakage from the test set into scaling/encoding statistics.


In [21]:
TARGET_COLUMN = "Churn"
NUMERIC_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]

y = (df[TARGET_COLUMN] == "Yes").astype(int)
X = df.drop(columns=[TARGET_COLUMN])
categorical_features = [c for c in X.columns if c not in NUMERIC_FEATURES]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train churn rate:", y_train.mean(), " Test churn rate:", y_test.mean())


Train: (5634, 19)  Test: (1409, 19)
Train churn rate: 0.2653532126375577  Test churn rate: 0.2654364797728886


In [22]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), categorical_features),
])


## 4. Model Training & Metric Choice

**Metric choice: ROC-AUC as primary, PR-AUC as secondary check.**

The required output (`risk_score`) is explicitly a *continuous ranking*, not
just a label — the assignment's own use cases ("which customers are most
likely to churn," "aggregate churn risk across segments") are ranking and
scoring tasks. ROC-AUC measures ranking quality (the probability a random
churner is scored higher than a random non-churner) independent of any
decision threshold, which fits that use case directly.

Plain **accuracy is rejected** as the primary metric: with ~26.5% positive
class, a trivial always-predict-"No" model scores ~73.5% accuracy while
being useless — accuracy would misrepresent model quality here.

ROC-AUC alone can look optimistic under class imbalance, so **PR-AUC
(average precision)** is reported alongside it as an honest cross-check —
it's more sensitive to performance on the minority (churn) class, which is
the class that actually matters operationally.

A fixed 0.5 classification threshold is also reported for reference
(precision/recall/confusion matrix), but the shipped `risk_score` is
designed to be used as a continuous score by the agent (e.g., for ranking
"most at risk" customers), not just a binary label — so threshold choice is
a downstream/business decision, not baked into the model itself.


In [23]:
def evaluate(pipe, X_test, y_test):
    proba = pipe.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    return {
        "roc_auc": roc_auc_score(y_test, proba),
        "pr_auc": average_precision_score(y_test, proba),
        "confusion_matrix_at_0.5": confusion_matrix(y_test, preds),
        "classification_report_at_0.5": classification_report(y_test, preds, digits=3),
    }


In [24]:
logreg = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
])
logreg.fit(X_train, y_train)
logreg_metrics = evaluate(logreg, X_test, y_test)

print("=== Logistic Regression ===")
print("ROC-AUC:", logreg_metrics["roc_auc"])
print("PR-AUC :", logreg_metrics["pr_auc"])
print(logreg_metrics["confusion_matrix_at_0.5"])
print(logreg_metrics["classification_report_at_0.5"])


=== Logistic Regression ===
ROC-AUC: 0.8417034798108968
PR-AUC : 0.632823667353558
[[748 287]
 [ 81 293]]
              precision    recall  f1-score   support

           0      0.902     0.723     0.803      1035
           1      0.505     0.783     0.614       374

    accuracy                          0.739      1409
   macro avg      0.704     0.753     0.708      1409
weighted avg      0.797     0.739     0.753      1409



In [25]:
rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced",
                                    random_state=RANDOM_STATE, n_jobs=-1)),
])
rf.fit(X_train, y_train)
rf_metrics = evaluate(rf, X_test, y_test)

print("=== Random Forest ===")
print("ROC-AUC:", rf_metrics["roc_auc"])
print("PR-AUC :", rf_metrics["pr_auc"])
print(rf_metrics["confusion_matrix_at_0.5"])
print(rf_metrics["classification_report_at_0.5"])


=== Random Forest ===
ROC-AUC: 0.842140070784572
PR-AUC : 0.6508422305849386
[[786 249]
 [ 90 284]]
              precision    recall  f1-score   support

           0      0.897     0.759     0.823      1035
           1      0.533     0.759     0.626       374

    accuracy                          0.759      1409
   macro avg      0.715     0.759     0.724      1409
weighted avg      0.801     0.759     0.770      1409



In [26]:
print("Majority-class baseline accuracy (reference only, not the chosen metric):",
      max(y_test.mean(), 1 - y_test.mean()))


Majority-class baseline accuracy (reference only, not the chosen metric): 0.7345635202271115


## 5. Model Selection

**Decision: ship Logistic Regression.**

The two models are close: Random Forest is marginally ahead on both ROC-AUC
(by roughly 0.0005) and PR-AUC (by roughly 0.018) — see the printed metrics
above for the exact run values. That gap is small enough that predictive
performance isn't the deciding factor here.

What *does* decide it is the required output shape: `predict_churn_risk()`
must return `top_factors`, i.e. a genuine per-prediction explanation. For a
linear model, `logit = intercept + Σ(coefficient_i × encoded_value_i)`, so
each encoded feature's exact signed contribution to *that specific*
prediction can be computed directly — no extra library, no approximation,
cheap enough to run inside a batch/aggregate query the agent might issue
repeatedly. Getting a comparable per-prediction explanation out of the
Random Forest would require SHAP or permutation importance — extra
dependency surface and materially more compute per call, for a predictive
gain of about two hundredths of a point on PR-AUC. Given the assignment
explicitly deprioritizes ML sophistication in favor of the reasoning behind
the choice, the interpretability trade-off is the right one here, and it's
a documented choice rather than a default.


In [27]:
prep = logreg.named_steps["prep"]
clf = logreg.named_steps["clf"]
feat_names = prep.get_feature_names_out()
coefs = clf.coef_[0]
coef_series = pd.Series(coefs, index=feat_names).sort_values()

print("Top factors DECREASING churn risk (global coefficients):")
print(coef_series.head(8))
print()
print("Top factors INCREASING churn risk (global coefficients):")
print(coef_series.tail(8))


Top factors DECREASING churn risk (global coefficients):
num__tenure                                 -1.156122
cat__Contract_Two year                      -0.811179
cat__InternetService_DSL                    -0.642560
num__MonthlyCharges                         -0.638849
cat__OnlineSecurity_No internet service     -0.301138
cat__StreamingTV_No internet service        -0.301138
cat__DeviceProtection_No internet service   -0.301138
cat__InternetService_No                     -0.301138
dtype: float64

Top factors INCREASING churn risk (global coefficients):
cat__OnlineSecurity_No                 0.166324
cat__PaymentMethod_Electronic check    0.216101
cat__StreamingTV_Yes                   0.220122
cat__StreamingMovies_Yes               0.232890
cat__PaperlessBilling_Yes              0.334308
num__TotalCharges                      0.491944
cat__Contract_Month-to-month           0.625314
cat__InternetService_Fiber optic       0.656003
dtype: float64


These global coefficients line up with the EDA in Section 1.3: long
contracts and DSL internet push risk down, month-to-month contracts and
fiber-optic internet push it up. This cross-check between independent EDA
and the fitted model's own coefficients is a useful sanity check that the
model learned real signal.

Note: these are *global* coefficients (one direction per feature, dataset-
wide). The `top_factors` field returned by `predict_churn_risk()` in
`src/model/predict.py` is a *per-prediction* explanation — it multiplies
each encoded feature's value for one specific customer by its coefficient,
so the ranked factors can differ from customer to customer (e.g. `tenure`
only shows up as a top factor for a given prediction if that customer's
tenure is unusually high or low).


## 6. Export Artifacts

Exports the fitted pipeline (preprocessing + shipped model), the cleaned
reference dataset (for `customer_id` lookups), and the evaluation metrics —
everything `src/model/predict.py` needs to run `predict_churn_risk()`
outside of this notebook.


In [28]:
os.makedirs("data", exist_ok=True)

artifact = {
    "pipeline": logreg,
    "model_name": "logistic_regression",
    "feature_columns": list(X.columns),
    "categorical_features": categorical_features,
    "numeric_features": NUMERIC_FEATURES,
}
joblib.dump(artifact, "data/model_pipeline.joblib")
df.to_csv("data/cleaned_churn.csv")

metrics_out = {
    "logistic_regression": {
        "roc_auc": logreg_metrics["roc_auc"],
        "pr_auc": logreg_metrics["pr_auc"],
    },
    "random_forest": {
        "roc_auc": rf_metrics["roc_auc"],
        "pr_auc": rf_metrics["pr_auc"],
    },
    "shipped_model": "logistic_regression",
}
with open("data/metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

print("Saved:")
print(" - data/model_pipeline.joblib")
print(" - data/cleaned_churn.csv")
print(" - data/metrics.json")


Saved:
 - data/model_pipeline.joblib
 - data/cleaned_churn.csv
 - data/metrics.json


## 7. Verify the Standalone Callable

Sanity check that the exported artifact works through the same
`predict_churn_risk()` function the Streamlit app / agent will call in
later stages (`src/model/predict.py`), not just inside this notebook's
in-memory objects.


In [29]:
import sys
sys.path.insert(0, ".")  # if running from the project root

from src.model.predict import predict_churn_risk

sample_id = df.index[0]
result = predict_churn_risk(customer_id=sample_id)
print("customer_id:", sample_id)
print("risk_score:", result["risk_score"])
print("top_factors:")
for f in result["top_factors"]:
    print(" ", f)


customer_id: 7590-VHVEG
risk_score: 0.8062756315136871
top_factors:
  {'feature': 'num__tenure', 'direction': 'increases_risk', 'contribution': 1.4817137915169065}
  {'feature': 'num__MonthlyCharges', 'direction': 'increases_risk', 'contribution': 0.7436696275845802}
  {'feature': 'cat__InternetService_DSL', 'direction': 'decreases_risk', 'contribution': -0.6425595296578692}
  {'feature': 'cat__Contract_Month-to-month', 'direction': 'increases_risk', 'contribution': 0.6253136444884951}
  {'feature': 'num__TotalCharges', 'direction': 'decreases_risk', 'contribution': -0.4898897237199934}
